# Makemore Part 2 — FULL Pipeline (MPS Optimized)
End-to-end: data → vocab → dataset → model → training → sampling

In [12]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import re
import random

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print('Using device:', device)

Using device: mps


## 1. Load & Clean Data

In [38]:
# Replace with your CSV if needed
import pandas as pd
df = pd.read_csv('arabnames.csv')

# Example fallback
names = df['Name'].str.lower()

names = names.apply(lambda x: re.sub(r'[^a-z]', '', x))
names = names.tolist()
# Clean (only a-z)
names = [re.sub(r'[^a-z]', '', w.lower()) for w in names]
names = [w for w in names if len(w)>0]

print(names[:5])

len(names)


['aaban', 'aabid', 'aadil', 'aahil', 'aalam']


4512

## 2. Build Vocabulary

In [19]:
chars = sorted(list(set(''.join(names))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

vocab_size = len(stoi)
print('vocab_size:', vocab_size)

vocab_size: 27


## 3. Build Dataset

In [20]:
block_size = 3

X, Y = [], []

for w in names:
    context = [0]*block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

print(X.shape, Y.shape)

torch.Size([32679, 3]) torch.Size([32679])


## 4. Split Dataset

In [21]:
n1 = int(0.8*len(X))
n2 = int(0.9*len(X))

Xtr, Ytr = X[:n1], Y[:n1]
Xdev, Ydev = X[n1:n2], Y[n1:n2]
Xte, Yte = X[n2:], Y[n2:]

## 5. Initialize Model

In [22]:
n_embd = 10
n_hidden = 100

C = torch.randn((vocab_size, n_embd), device=device)
W1 = torch.randn((n_embd*block_size, n_hidden), device=device)
b1 = torch.randn(n_hidden, device=device)
W2 = torch.randn((n_hidden, vocab_size), device=device)
b2 = torch.randn(vocab_size, device=device)

parameters = [C, W1, b1, W2, b2]

for p in parameters:
    p.requires_grad = True

Xtr = Xtr.to(device)
Ytr = Ytr.to(device)

## 6. Training

In [23]:
batch_size = 1024
max_steps = 50000

for i in range(max_steps):
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), device=device)

    emb = C[Xtr[ix]]
    emb = emb.reshape(batch_size, -1)

    h = torch.tanh(emb @ W1 + b1)
    logits = h @ W2 + b2

    loss = F.cross_entropy(logits, Ytr[ix])

    for p in parameters:
        p.grad = None
    loss.backward()

    lr = 0.1 if i < 30000 else 0.01

    for p in parameters:
        p.data += -lr * p.grad

    if i % 1000 == 0:
        print(i, loss.item())

0 17.578767776489258
1000 3.096250534057617
2000 2.4187052249908447
3000 2.3893625736236572
4000 2.116090774536133
5000 2.1816797256469727
6000 2.115190029144287
7000 2.137986183166504
8000 2.0027854442596436
9000 2.079847574234009
10000 2.0457303524017334
11000 1.9341845512390137
12000 2.0048704147338867
13000 1.9491113424301147
14000 1.898822546005249
15000 1.9779006242752075
16000 1.936598300933838
17000 1.9035975933074951
18000 1.9222838878631592
19000 1.8878514766693115
20000 1.9263222217559814
21000 1.887911319732666
22000 1.8424785137176514
23000 1.9265227317810059
24000 1.935999870300293
25000 1.8888260126113892
26000 1.898171305656433
27000 1.8594825267791748
28000 1.9097704887390137
29000 1.8964565992355347
30000 1.8679442405700684
31000 1.8884285688400269
32000 1.8504676818847656
33000 1.8174588680267334
34000 1.9312245845794678
35000 1.8355991840362549
36000 1.8186942338943481
37000 1.8662490844726562
38000 1.8497843742370605
39000 1.8752994537353516
40000 1.790618777275085

## 7. Sampling

In [37]:
g = random.seed()

for _ in range(100):
    out = []
    context = [0]*block_size

    while True:
        emb = C[torch.tensor([context], device=device)]
        emb = emb.reshape(1, -1)

        h = torch.tanh(emb @ W1 + b1)
        logits = h @ W2 + b2

        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()

        context = context[1:] + [ix]
        if ix == 0:
            break
        out.append(itos[ix])

    print(''.join(out))

khawid
waylr
kameed
nadiyaha
haimah
kafi
surat
must
kharafwahir
mal
sabadi
kaid
ewaheeda
mawaq
huwnia
nim
aneeq
alyan
atim
radad
far
shaf
tah
han
hasanan
abdulwar
ari
abdullabadan
faas
batahal
jabeeb
dawrah
mayudayaniya
dah
nam
baseem
khalees
fawt
lar
ohufayoob
is
izzz
naser
nasood
eldin
jebrar
abdulwaid
eldat
hudra
mineaaziz
munjwaaiyusakwi
aamal
kha
eima
bulquraiyan
jah
las
man
hameelah
wabsha
mahhid
abdulria
boul
alil
nishadir
baq
tadir
abdulwazser
naqeel
kah
ayu
luwaz
aleil
bahir
akaalia
nadwu
jasyana
ashrtadiq
tallabaalik
sular
talia
sah
nahlrah
azim
fadi
tafi
zafadilah
iryad
mayr
abdulkalim
fik
baredah
murshamaaldin
asheem
gohah
larl
asi
azm
asrf
yushar
